# PlasticWatch — Train YOLO11 on Roboflow Plastic Waste Dataset

This Google Colab notebook automates the end-to-end training of Ultralytics YOLO11 on the **Plastic Waste Dataset** (12,484 images) from Roboflow Universe.

### Dataset Provenance
- **Source:** [Roboflow Universe: Plastic Waste Dataset (v2)](https://universe.roboflow.com/edwin-daza-saavedra-s-workspace/plastic-waste-ag4eg/dataset/2#)
- **Classes:** `plastic bottle`, `plastic bag`, `plastic cup`
- **Total Images:** 12,484 (Train: 9,609, Valid: 1,983, Test: 892)
- **License:** CC BY 4.0

```bibtex
@misc{ plastic-waste-ag4eg_dataset,
  title = { Plastic Waste Dataset },
  type = { Open Source Dataset },
  author = { Edwin Daza Saavedra's Workspace },
  howpublished = { \url{ https://universe.roboflow.com/edwin-daza-saavedra-s-workspace/plastic-waste-ag4eg } },
  url = { https://universe.roboflow.com/edwin-daza-saavedra-s-workspace/plastic-waste-ag4eg },
  journal = { Roboflow Universe },
  publisher = { Roboflow },
  year = { 2026 },
  month = { jul },
  note = { visited on 2026-09-26 },
}
```

## 1. Environment & GPU Verification

In [ ]:
# Verify GPU runtime (Runtime -> Change runtime type -> T4 GPU / A100 GPU)
!nvidia-smi

# Install dependencies
!pip install -q "ultralytics>=8.3.0" roboflow

## 2. Download Dataset from Roboflow Universe

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "741YgPzdJ9QttK3uR4BG"
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace("edwin-daza-saavedra-s-workspace").project("plastic-waste-ag4eg")
version = project.version(2)
dataset = version.download("yolov11")

print("Dataset downloaded at:", dataset.location)
!cat {dataset.location}/data.yaml

## 3. Train YOLO11s on Plastic Waste Dataset

In [ ]:
from ultralytics import YOLO

# Initialize base model (YOLO11s pretrained)
model = YOLO("yolo11s.pt")

# Train on GPU
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    save=True,
    project="runs/detect",
    name="plastic_waste_yolo11s",
    seed=42,
    deterministic=True
)

print("Training complete! Best weights saved at:", results.save_dir + "/weights/best.pt")

## 4. Validate Model & Print Per-Class Metrics

In [ ]:
best_weights = f"{results.save_dir}/weights/best.pt"
val_model = YOLO(best_weights)
metrics = val_model.val(data=f"{dataset.location}/data.yaml", split="test")

print("\n--- Test Set Metrics ---")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 5. Download `best.pt` to Deploy into PlasticWatch

Run this cell to download `best.pt` directly to your computer. Place the downloaded file into:
`plasticwatch/backend/weights/best.pt`.

In [ ]:
from google.colab import files
import os

best_pt_path = f"{results.save_dir}/weights/best.pt"
if os.path.exists(best_pt_path):
    print(f"Downloading {best_pt_path} ({os.path.getsize(best_pt_path)/(1024*1024):.1f} MB)...")
    files.download(best_pt_path)
else:
    print("File not found! Check results.save_dir")

## 6. (Optional) Deploy Directly to Roboflow Serverless Cloud

In [ ]:
# To deploy to Roboflow hosted inference API:
# version.deploy(model_type="yolov11", model_path=f"{results.save_dir}")